In [12]:
import os
import numpy as np
import pandas as pd
import json
import systeme as sys
import Allocation
from simulation import simulation
import time

import pickle
import random
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import optuna
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, make_scorer, hamming_loss
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.base import BaseEstimator
from functools import partial

import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

instance_name = "G"

if instance_name == "K0":
    test_split_scenarios = [23, 16, 7, 10, 26, 17, 6, 20, 11]
    fms_path = 'fms/3C7R5F.json'
    nombre_de_cellules = 3
    nombre_de_scenarios = 28
else :
    test_split_scenarios = [37, 13, 31, 40, 25, 14, 6, 12, 4, 7, 28, 15, 17]
    fms_path = 'fms/5C14R5F.json'
    nombre_de_cellules = 5
    nombre_de_scenarios = 40

with open(fms_path, 'r') as json_file: 
    dic = json.load(json_file)
system = sys.systeme(dic)

## entraienement

### multilabel

In [13]:
def calcul_gap(modele, scaler, cellule, scenarios):
    
    scenarios_path = f"scenarios/{instance_name}"
    solution_path = f"solution/{instance_name}_upgraded//"
    
    with open(fms_path, 'r') as json_file:
        dic = json.load(json_file)
    s = sys.systeme(dic)
    all_gaps = []
    for f in os.listdir(scenarios_path):
        if int(f.split(".")[0][1:]) not in scenarios:
            continue
        df = pd.DataFrame(np.nan_to_num(pd.read_csv(f"{scenarios_path}/"+f,header=None, index_col=None, sep=";"), nan=0)).astype(int)
        own_sol_path_list = [file for file in os.listdir(solution_path) if file.split(".")[0][-len(f.split(".")[0]):] == f.split(".")[0]]
        if len(own_sol_path_list) == 0:
            continue
        sol_path = own_sol_path_list[0]
        sol = pd.read_csv(solution_path+sol_path, sep=";", index_col=None, header=None).iloc[:,1:-1]

        allocators = [Allocation.StaticAllocator(s, sol) for _ in range(len(s.cellules))]
        ref_mct = simulation(system=s, scenario=df, allocators=allocators).mean_completion_time()
        allocators[cellule] = Allocation.DynamicAllocator(s, Allocation.GlobalMultiLabel(modele, s, cellule), scaler, to_categorical=True)
        mct = simulation(system=s, scenario=df, allocators=allocators).mean_completion_time()
        gap = (100*(mct - ref_mct)/ref_mct)
        all_gaps.append(gap)

    return np.mean(all_gaps)

#multilabel
def callback(_, trial, cell, X_train, Y_train, scaler, scenarios_test, scores, cv):
    current_params = trial.params
    current_model = RandomForestClassifier(**current_params, random_state=RANDOM_SEED)
    current_model.fit(X_train, Y_train)

    current_gap = calcul_gap(current_model, scaler, cell, scenarios_test)
    acc = cross_val_score(current_model, X_train, Y_train, cv=cv, scoring=make_scorer(element_wise_accuracy_rf)).mean()
    #scores.append(acc)
    scores.append([current_gap, acc])
    
#multilabel
def element_wise_accuracy_rf(y_true, y_pred):
    # Convertir en numpy array pour éviter les conflits avec pandas
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return float((y_true == y_pred).mean())

#mutlilabel
def objective_rf(trial, X_train, y_train, cv=3, scenarios_test=None, scaler=None, cell=None):
    """
    Fonction objectif pour optimiser les hyperparamètres d'un Random Forest avec Optuna.
    """
    # Définir les hyperparamètres à optimiser
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 1, 20)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])

    # Initialiser le modèle avec les hyperparamètres
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=RANDOM_SEED
    )

    # Effectuer une validation croisée pour évaluer la performance

    model.fit(X_train, y_train)

    return  -calcul_gap(model, scaler, cell, scenarios_test)
    #scores = cross_val_score(model, X_train, y_train, cv=cv, scoring=make_scorer(element_wise_accuracy_rf))
    #return scores.mean()  # Retourner la moyenne des scores comme métrique


def train_random_forest_with_optuna(X_train, y_train, X_test, y_test, scaler, max_trials=49, alpha=50, beta=0.1, patience_limit=1, random_seed=RANDOM_SEED, cell=1, scenarios_test=[]):
    """
    Entraîner un Random Forest optimisé via une recherche bayésienne sur les hyperparamètres.
    """
    # Initialiser une étude Optuna
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=random_seed))
    total_trials = 0
    remaining_trials = alpha  # Nombre initial de trials
    step = 0
    best_known_score = 0
    patience = 0
    scores = []

    custom_callback = partial(callback, X_train=X_train, Y_train=y_train, scenarios_test=scenarios_test, scores=scores, cell=cell, scaler=scaler, cv=3)

    while total_trials < max_trials and patience < patience_limit:
        print(f"Optimizing: Step {step + 1}, Remaining trials: {remaining_trials}")
        study.optimize(lambda trial: objective_rf(trial, X_train, y_train, cv=3, scenarios_test=scenarios_test, scaler=scaler, cell=cell), n_trials=remaining_trials, callbacks=[custom_callback])
        
        remaining_trials = int(np.ceil(alpha / (1 + beta * step)))
        total_trials += remaining_trials
        step += 1

        best_current_score = study.best_value

        if best_current_score > best_known_score:
            best_known_score = best_current_score
            patience = 0 
        else:
            patience += 1  

    # Afficher les meilleurs paramètres trouvés
    best_params = study.best_params
    print(f"Best Hyperparameters: {best_params}")

    # Entraîner le modèle avec les meilleurs hyperparamètres
    model = RandomForestClassifier(**best_params, random_state=random_seed)
    model.fit(X_train, y_train)

    # Prédire sur le jeu de test et afficher les performances
    predictions = model.predict(X_test)
    print("Optimized Random Forest - Classification Report:")
    print(classification_report(y_test, predictions))

    return model, best_params


In [14]:
with open(fms_path, 'r') as json_file: #preparation de donnees
    dic = json.load(json_file)
s = sys.systeme(dic)

data_folder_path = f"generated_data/{instance_name}/multilabel/final/"
test_scenario_count = nombre_de_scenarios // 3

file_names_per_cell, columns_per_cell, dfs_train, dfs_test = [],[],[],[]


for cell,c in zip(s.cellules,range(len(s.cellules))):
    file_names_per_cell.append([file for file in os.listdir(data_folder_path)[:] if file.endswith(f"cell_{c+1}.csv")])
    kept_cols = cell.header[:-1]
    columns_per_cell.append(kept_cols)
    
    if c == 0:
        test_split = list(range(1, len(file_names_per_cell[0])+1))
        random.shuffle(test_split)
        test_split = test_split[:test_scenario_count]
        test_split_scenarios = test_split[:len(test_split)]
        train_split_scenarios = [file for file in list(range(1,len(file_names_per_cell[0])+1)) if file not in test_split]

    paths = sorted([data_folder_path+"/"+x for x in os.listdir(data_folder_path) if x.endswith(f"{c+1}.csv")], key= lambda k: int(k.split("_cell_")[0].split("s")[-1]))

    filtered_dfs_train = [pd.read_csv(paths[scenar-1], delimiter=";") for scenar in train_split_scenarios]
    filtered_dfs_test = [pd.read_csv(paths[scenar-1], delimiter=";") for scenar in test_split_scenarios]
    

    df_train = pd.concat(filtered_dfs_train, ignore_index=True)
    df_train.fillna(0, inplace=True)


    categorical_cols = [col for col in df_train.columns if col.startswith("Family")]
    to_delete_cols = [col+"_0" for col in df_train.columns if col.startswith("Family")]  
    last_col = [col for col in df_train.columns if col.startswith("Selected")]
    categorical_spec = {
        "Family": [1, 2, 3, 4, 5]
        }
    new_columns = []
    for prefix, unique_values in categorical_spec.items():
        cols_to_encode = [col for col in df_train.columns if col.startswith(prefix)]
        for col in cols_to_encode:
            for value in unique_values:
                new_col_name = f"{col}_{value}"
                new_columns.append(pd.DataFrame({new_col_name: (df_train[col] == value).astype(float)}))
    df_train = pd.concat([df_train] + new_columns, axis=1)       
    df_train.drop(columns=to_delete_cols, inplace=True, errors='ignore')
    df_train.drop(columns=categorical_cols, inplace=True, errors='ignore')

    colus = df_train.columns.tolist()
    for col in last_col:
        colus.remove(col)
        df_train = df_train[colus + [col]]
        colus.append(col)
    
    no_label_cols = [col for col in df_train.columns if not col.startswith("Selected")]
    label_cols = [col for col in df_train.columns if col.startswith("Selected")]

    dfs_train.append(df_train.astype(float))



    df_test = pd.concat(filtered_dfs_test, ignore_index=True)
    df_test.fillna(0, inplace=True)

    new_columns = []
    for prefix, unique_values in categorical_spec.items():
        cols_to_encode = [col for col in df_test.columns if col.startswith(prefix)]
        for col in cols_to_encode:
            for value in unique_values:
                new_col_name = f"{col}_{value}"
                new_columns.append(pd.DataFrame({new_col_name: (df_test[col] == value).astype(float)}))
    df_test = pd.concat([df_test] + new_columns, axis=1)
    df_test.drop(columns=to_delete_cols, inplace=True, errors='ignore')
    df_test.drop(columns=categorical_cols, inplace=True, errors='ignore')

    colus = df_test.columns.tolist()
    for col in last_col:
        colus.remove(col)
        df_test = df_test[colus + [col]]
        colus.append(col)

    dfs_test.append(df_test.astype(float))


### models training

In [15]:
#training (les deux datasets le meme code, changer juste point rouge)

models = []

for i, (train_df, test_df) in enumerate(zip(dfs_train, dfs_test)):

    print(f"\nProcessing dataset {i+1}...")
    nb_classes = len([col for col in train_df.columns if col.startswith("Selected")])
    print(f"\nNumber of ressources {nb_classes}...")

    X_train, y_train = train_df.iloc[:, :-nb_classes], train_df.iloc[:, -nb_classes:].map(lambda x: 1 if x > 0 else 0)
    X_test, y_test = test_df.iloc[:, :-nb_classes], test_df.iloc[:, -nb_classes:].map(lambda x: 1 if x > 0 else 0)

    model = train_random_forest_with_optuna(X_train, y_train, X_test, y_test, None, max_trials = 500, alpha= 100, beta = 0.1, random_seed=RANDOM_SEED, scenarios_test=test_split_scenarios,cell=i)
    
    predictions_test = model[0].predict(X_test)
    print(f"Validation Performance for dataset {i+1}:\n",classification_report(y_test, predictions_test))
    print(y_test.shape, predictions_test.shape)
    accuracy = np.mean(y_test == predictions_test)
    print(f"Accuracy: {accuracy}")
    print(f"Hamming Score: {1 - hamming_loss(y_test, predictions_test)}")
    
    predicted = np.argmax(np.column_stack([model[0].predict_proba(X_test)[i][:,1] for i in range(len(model[0].predict_proba(X_test)))]), axis=1)
    count = 0
    for j in range(predicted.shape[0]):
        ress_choice = predicted[j] 
        if y_test.iloc[j, ress_choice] == 1:  
            count += 1
    print(f"calculated accuracy based on top probability only:{count/predicted.shape[0]}")

    print("- - - saving - - -")
    models.append(model[0])
    with open(f"generated_models/{instance_name}/multilabel/models_cell{i+1}/standard_RandomForest.pkl", 'wb') as file: #change G for K0
        pickle.dump(model[0], file)
    

[I 2025-07-18 18:56:23,644] A new study created in memory with name: no-name-387129de-138b-4497-ada7-3249f32484ca



Processing dataset 1...

Number of ressources 2...
Optimizing: Step 1, Remaining trials: 100


[I 2025-07-18 18:56:29,632] Trial 0 finished with value: -38.92624105443789 and parameters: {'n_estimators': 144, 'max_depth': 20, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 'sqrt'}. Best is trial 0 with value: -38.92624105443789.
[I 2025-07-18 18:56:44,796] Trial 1 finished with value: -24.530819473864945 and parameters: {'n_estimators': 267, 'max_depth': 13, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: -24.530819473864945.
[I 2025-07-18 18:57:00,074] Trial 2 finished with value: -56.23177493180424 and parameters: {'n_estimators': 95, 'max_depth': 4, 'min_samples_split': 7, 'min_samples_leaf': 11, 'max_features': None}. Best is trial 1 with value: -24.530819473864945.
[I 2025-07-18 18:57:14,830] Trial 3 finished with value: -31.096867407552438 and parameters: {'n_estimators': 85, 'max_depth': 6, 'min_samples_split': 8, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 1 with value: -24.530819473

Best Hyperparameters: {'n_estimators': 241, 'max_depth': 16, 'min_samples_split': 15, 'min_samples_leaf': 2, 'max_features': 'sqrt'}


[I 2025-07-18 19:38:10,059] A new study created in memory with name: no-name-42c2205d-0c00-4da9-a1e5-ae7730c35785


...

Best Hyperparameters: {'n_estimators': 178, 'max_depth': 15, 'min_samples_split': 20, 'min_samples_leaf': 18, 'max_features': 'log2'}


[I 2025-07-18 21:34:49,285] A new study created in memory with name: no-name-96b7f254-4f65-4b43-b15d-f3bc3d571557


Optimized Random Forest - Classification Report:
              precision    recall  f1-score   support

           0       0.86      1.00      0.92      1114
           1       0.86      1.00      0.92      1116
           2       0.85      1.00      0.92      1105

   micro avg       0.86      1.00      0.92      3335
   macro avg       0.86      1.00      0.92      3335
weighted avg       0.86      1.00      0.92      3335
 samples avg       0.86      1.00      0.91      3335

Validation Performance for dataset 4:
               precision    recall  f1-score   support

           0       0.86      1.00      0.92      1114
           1       0.86      1.00      0.92      1116
           2       0.85      1.00      0.92      1105

   micro avg       0.86      1.00      0.92      3335
   macro avg       0.86      1.00      0.92      3335
weighted avg       0.86      1.00      0.92      3335
 samples avg       0.86      1.00      0.91      3335

(1300, 3) (1300, 3)
Accuracy: 0.8551282051

[I 2025-07-18 21:34:57,137] Trial 0 finished with value: -0.3025153172962407 and parameters: {'n_estimators': 144, 'max_depth': 20, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 'sqrt'}. Best is trial 0 with value: -0.3025153172962407.
[I 2025-07-18 21:35:14,525] Trial 1 finished with value: -0.31751967837827483 and parameters: {'n_estimators': 267, 'max_depth': 13, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: -0.3025153172962407.
[I 2025-07-18 21:35:35,936] Trial 2 finished with value: -0.31541918303289385 and parameters: {'n_estimators': 95, 'max_depth': 4, 'min_samples_split': 7, 'min_samples_leaf': 11, 'max_features': None}. Best is trial 0 with value: -0.3025153172962407.
[I 2025-07-18 21:35:55,573] Trial 3 finished with value: -0.342229203727611 and parameters: {'n_estimators': 85, 'max_depth': 6, 'min_samples_split': 8, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 0 with value: -0.302515

Best Hyperparameters: {'n_estimators': 263, 'max_depth': 15, 'min_samples_split': 6, 'min_samples_leaf': 6, 'max_features': None}
Optimized Random Forest - Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.80      0.80       516
           1       0.88      0.87      0.88       811

   micro avg       0.85      0.84      0.85      1327
   macro avg       0.84      0.84      0.84      1327
weighted avg       0.85      0.84      0.85      1327
 samples avg       0.85      0.85      0.85      1327

Validation Performance for dataset 5:
               precision    recall  f1-score   support

           0       0.79      0.80      0.80       516
           1       0.88      0.87      0.88       811

   micro avg       0.85      0.84      0.85      1327
   macro avg       0.84      0.84      0.84      1327
weighted avg       0.85      0.84      0.85      1327
 samples avg       0.85      0.85      0.85      1327

(1300, 2) (1300, 2)


## Simulation

In [17]:
with open(fms_path, 'r') as json_file: #5C14R5F #3C7R5F
    dic = json.load(json_file)
s = sys.systeme(dic)

models= []
model_classes= []
for i in range(1,nombre_de_cellules+1):
    with open(f"generated_models/{instance_name}/multilabel/models_cell{i}/standard_RandomForest.pkl", 'rb') as f:
        m = pickle.load(f)
        models.append(m)
        model_classes.append(Allocation.GlobalMultiLabel(m, s, i))

scenarios_path = f"scenarios/{instance_name}"
solution_path = f"solution/{instance_name}_upgraded/"

results = np.zeros((0,4))
reference_results = np.zeros((0,4))

for f in sorted(os.listdir(scenarios_path), key=lambda y: int(y.split(".")[0][1:])):
    #print(f"{f.split(".")[0]}\t:\t", end = "")
    #if int(f.split(".")[0][1:]) not in test_split_scenarios:
    #    print("dans les scenarios d'entrainement",end="\r")
    #    continue
    df = pd.DataFrame(np.nan_to_num(pd.read_csv(f"{scenarios_path}/"+f, index_col=None, header=None, sep=";"), nan=0)).astype(int)
    own_sol_path_list = [file for file in os.listdir(solution_path) if file.split(".")[0][-len(f.split(".")[0]):] == f.split(".")[0]]
    
    sol_path = own_sol_path_list[0]
    sol = pd.read_csv(solution_path+sol_path, sep=";", index_col=None, header=None).iloc[:,1:-1]

    dyn_allocs = [Allocation.DynamicAllocator(s, model_class, None, keep_cols=None, to_categorical=True, singlelabel=True) for model_class in model_classes]
    
    static_allocators = [Allocation.StaticAllocator(s, sol) for _ in range(nombre_de_cellules)]
    
    allocators = [#static_allocators[0],
                  dyn_allocs[0], 
                  #static_allocators[1], 
                  dyn_allocs[1], 
                  #static_allocators[2], 
                  dyn_allocs[2]
                  ] if instance_name == "K0" else [
                      #static_allocators[0],
                  dyn_allocs[0], 
                  #static_allocators[1], 
                  dyn_allocs[1], 
                  #static_allocators[2], 
                  dyn_allocs[2], 
                  #static_allocators[3], 
                  dyn_allocs[3], 
                  #static_allocators[4], 
                  dyn_allocs[4]
                  ]
    
    sim = simulation(system=s, scenario=df, allocators=allocators)
    results = np.vstack((results, np.array([f.split(".")[0], sim.average_flowtime(), sim.mean_completion_time(), sim.total_decision_times()])))

    sim_ref = simulation(system=s, scenario=df, allocators=static_allocators)
    reference_results = np.vstack((reference_results, np.array([f.split(".")[0], sim_ref.average_flowtime(), sim_ref.mean_completion_time(), sim_ref.total_decision_times()])))
 
    print(f"{f.split(".")[0]}\t{sim.mean_completion_time()}\t{sim_ref.mean_completion_time()}\t{int(f.split(".")[0][1:]) in test_split_scenarios}")
    #print(f"ended with mct : {sim.mean_completion_time()} while the reference mct is {sim_ref.mean_completion_time()}")

    #sim_ref.gantt(path=f"gants/gant_{f.split(".")[0]}_ref.png")
    #sim.gantt(path=f"gants/gant_{f.split(".")[0]}.png")

df = pd.DataFrame(results, columns=["scenario", "avg_flowtime", "mct", "decision_time"])
df['num'] = df['scenario'].str.extract(r'(\d+)').astype(int)
df_sorted = df.sort_values(by='num').drop(columns='num').reset_index(drop=True).mct.astype(float)

df_ref = pd.DataFrame(reference_results, columns=["scenario", "avg_flowtime", "mct", "decision_time"])
df_ref['num'] = df['scenario'].str.extract(r'(\d+)').astype(int)
ref = df_ref.sort_values(by='num').drop(columns='num').reset_index(drop=True).mct.astype(float)

print(f"gap {"ag" if isinstance(allocators[0], Allocation.StaticAllocator) else "m"} {"ag" if isinstance(allocators[1], Allocation.StaticAllocator) else "m"} {"ag" if isinstance(allocators[2], Allocation.StaticAllocator) else "m"} {(100*(df_sorted - ref)/ref).mean()} %")

s1	353.81	296.5	False
s2	355.65	305.71	False
s3	359.91	303.49	False
s4	344.48	294.63	True
s5	362.19	296.63	True
s6	335.64	297.42	False
s7	338.6	298.24	False
s8	374.98	301.53	False
s9	359.92	312.66	False
s10	357.08	299.32	True
s11	363.65	303.6	True
s12	367.49	309.91	True
s13	352.47	289.79	False
s14	308.16	286.18	False
s15	338.12	290.94	False
s16	336.8	294.05	False
s17	352.96	312.46	False
s18	362.06	302.11	False
s19	368.7	311.99	False
s20	349.44	293.85	False
s21	316.62	284.78	False
s22	362.68	306.75	True
s23	357.46	299.14	False
s24	302.86	292.74	True
s25	324.51	294.01	True
s26	304.16	280.28	True
s27	342.31	301.28	False
s28	317.28	290.57	False
s29	334.97	290.09	True
s30	299.01	287.15	False
s31	292.0	286.95	True
s32	337.69	292.29	False
s33	317.53	284.18	True
s34	346.42	299.02	False
s35	334.43	294.45	False
s36	325.57	287.62	False
s37	309.94	277.46	True
s38	333.28	294.8	False
s39	359.11	310.03	False
s40	334.72	288.4	False
gap m m m 14.718253024199322 %


## accuracy evals

In [ ]:
with open(fms_path, 'r') as json_file: #preparation de donnees
    dic = json.load(json_file)
s = sys.systeme(dic)

data_folder_path = f"generated_data/{instance_name}/multilabel/final/"
total_nb_scenarios = 40 * 3
test_scenario_count = 20

file_names_per_cell, columns_per_cell, dfs_train, dfs_test = [],[],[],[]


for cell,c in zip(s.cellules,range(len(s.cellules))):
    file_names_per_cell.append([file for file in os.listdir(data_folder_path)[:] if file.endswith(f"cell_{c+1}.csv")])
    kept_cols = cell.header[:-1]
    columns_per_cell.append(kept_cols)
    
    if c == 0:
        test_split = list(range(1, len(file_names_per_cell[0])+1))
        random.shuffle(test_split)
        test_split = test_split[:test_scenario_count]
        test_split_scenarios = test_split[:len(test_split)]
        train_split_scenarios = [file for file in list(range(1,len(file_names_per_cell[0])+1)) if file not in test_split]

    paths = sorted([data_folder_path+"/"+x for x in os.listdir(data_folder_path) if x.endswith(f"{c+1}.csv")], key= lambda k: int(k.split("_cell_")[0].split("s")[-1]))

    filtered_dfs_train = [pd.read_csv(paths[scenar-1], delimiter=";") for scenar in train_split_scenarios]
    filtered_dfs_test = [pd.read_csv(paths[scenar-1], delimiter=";") for scenar in test_split_scenarios]
    

    df_train = pd.concat(filtered_dfs_train, ignore_index=True)
    df_train.fillna(0, inplace=True)


    categorical_cols = [col for col in df_train.columns if col.startswith("Family")]
    to_delete_cols = [col+"_0" for col in df_train.columns if col.startswith("Family")]  
    last_col = [col for col in df_train.columns if col.startswith("Selected")]
    categorical_spec = {
        "Family": [1, 2, 3, 4, 5]
        }
    new_columns = []
    for prefix, unique_values in categorical_spec.items():
        cols_to_encode = [col for col in df_train.columns if col.startswith(prefix)]
        for col in cols_to_encode:
            for value in unique_values:
                new_col_name = f"{col}_{value}"
                new_columns.append(pd.DataFrame({new_col_name: (df_train[col] == value).astype(float)}))
    df_train = pd.concat([df_train] + new_columns, axis=1)       
    df_train.drop(columns=to_delete_cols, inplace=True, errors='ignore')
    df_train.drop(columns=categorical_cols, inplace=True, errors='ignore')

    colus = df_train.columns.tolist()
    for col in last_col:
        colus.remove(col)
        df_train = df_train[colus + [col]]
        colus.append(col)
    
    no_label_cols = [col for col in df_train.columns if not col.startswith("Selected")]
    label_cols = [col for col in df_train.columns if col.startswith("Selected")]

    dfs_train.append(df_train.astype(float))



    df_test = pd.concat(filtered_dfs_test, ignore_index=True)
    df_test.fillna(0, inplace=True)

    new_columns = []
    for prefix, unique_values in categorical_spec.items():
        cols_to_encode = [col for col in df_test.columns if col.startswith(prefix)]
        for col in cols_to_encode:
            for value in unique_values:
                new_col_name = f"{col}_{value}"
                new_columns.append(pd.DataFrame({new_col_name: (df_test[col] == value).astype(float)}))
    df_test = pd.concat([df_test] + new_columns, axis=1)
    df_test.drop(columns=to_delete_cols, inplace=True, errors='ignore')
    df_test.drop(columns=categorical_cols, inplace=True, errors='ignore')

    colus = df_test.columns.tolist()
    for col in last_col:
        colus.remove(col)
        df_test = df_test[colus + [col]]
        colus.append(col)

    dfs_test.append(df_test.astype(float))



models= []
for i in range(1,len(system.cellules)+1):
    with open(f"generated_models/{instance_name}/multilabel/models_cell{i}/standard_RandomForest.pkl", 'rb') as f:
        models.append(pickle.load(f))


In [36]:
for i, (model, train_df, test_df) in enumerate(zip(models, dfs_train, dfs_test)):

    print(f"\nEvaluating dataset {i+1}...")
    nb_classes = len([col for col in train_df.columns if col.startswith("Selected")])
    print(f"\nNumber of ressources {nb_classes}...")

    X_train, y_train = train_df.iloc[:, :-nb_classes], train_df.iloc[:, -nb_classes:].map(lambda x: 1 if x > 0 else 0)
    X_test, y_test = test_df.iloc[:, :-nb_classes], test_df.iloc[:, -nb_classes:].map(lambda x: 1 if x > 0 else 0)

    predictions_test = model.predict(X_test)
    print(f"Validation Performance for dataset {i+1}:\n",classification_report(y_test, predictions_test))
    print(y_test.shape, predictions_test.shape)
    accuracy = np.mean(y_test == predictions_test)
    print(f"Accuracy: {accuracy}")
    print(f"Hamming Score: {1 - hamming_loss(y_test, predictions_test)}")

    predicted = np.argmax(np.column_stack([model.predict_proba(X_test)[j][:,1] for j in range(len(model.predict_proba(X_test)))]), axis=1)
    count = 0
    for j in range(predicted.shape[0]):
        ress_choice = predicted[j] 
        if y_test.iloc[j, ress_choice] == 1:  
            count += 1
    print(f"calculated accuracy based on top probability only:{count/predicted.shape[0]}")


Evaluating dataset 1...

Number of ressources 2...
Validation Performance for dataset 1:
               precision    recall  f1-score   support

           0       0.78      0.87      0.82       393
           1       0.81      0.69      0.74       302

   micro avg       0.79      0.79      0.79       695
   macro avg       0.80      0.78      0.78       695
weighted avg       0.80      0.79      0.79       695
 samples avg       0.80      0.79      0.79       695

(689, 2) (689, 2)
Accuracy: 0.7910014513788098
Hamming Score: 0.7910014513788098
calculated accuracy based on top probability only:0.7968069666182874

Evaluating dataset 2...

Number of ressources 3...
Validation Performance for dataset 2:
               precision    recall  f1-score   support

           0       0.72      0.62      0.67       321
           1       0.73      0.44      0.55       300
           2       0.75      0.60      0.67       339

   micro avg       0.73      0.56      0.63       960
   macro avg   